# Jour 2 · Améliorer le retrieval avec des comparaisons


## Une valeur de chunk n'est pas une vérité universelle

`chunk_size=600` peut être raisonnable pour un corpus et mauvais pour un autre. La bonne taille dépend de la structure des documents, des questions et de la quantité de contexte nécessaire.

Nous allons comparer deux configurations sur les mêmes textes et les mêmes questions.


![Petits, moyens et grands chunks comparés](../assets/jour_01/09_comparer_chunking.png)

*Le choix doit être traité comme une hypothèse mesurable.*


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

CHAT_MODEL = "qwen2.5:3b"
EMBEDDING_MODEL = "bge-m3"

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm = ChatOllama(model=CHAT_MODEL, temperature=0)

print("Modèle de génération :", CHAT_MODEL)
print("Modèle d'embeddings  :", EMBEDDING_MODEL)


## Préparer un petit corpus catégorisé

La métadonnée `categorie` permettra ensuite de montrer un filtre exact. Elle décrit le document ; elle n'est pas calculée par la similarité.


In [ ]:
from langchain_core.documents import Document


documents = [
    Document(
        page_content=(
            "GlobalProtect permet l'accès distant. Le portail est vpn.entreprise.test. "
            "En cas d'erreur portail inaccessible, vérifier Internet puis l'adresse saisie. "
            "Si l'échec persiste, créer un ticket Réseau > VPN avec l'heure et une capture."
        ),
        metadata={"source": "procedure_vpn.md", "categorie": "reseau"},
    ),
    Document(
        page_content=(
            "Pour un écran noir, contrôler le voyant, l'alimentation et le câble vidéo. "
            "Choisir ensuite la bonne source HDMI ou DisplayPort. Si aucun affichage ne revient, "
            "créer un ticket Matériel > Affichage."
        ),
        metadata={"source": "procedure_ecran.md", "categorie": "materiel"},
    ),
    Document(
        page_content=(
            "Pour un bourrage papier, annuler l'impression puis éteindre l'imprimante. "
            "Retirer la feuille dans le sens de circulation et contrôler les rouleaux. "
            "Ne jamais utiliser d'objet métallique."
        ),
        metadata={"source": "procedure_imprimante.md", "categorie": "materiel"},
    ),
    Document(
        page_content=(
            "Un mot de passe et un code MFA sont secrets. Le support ne les demande jamais. "
            "Une demande inattendue doit être refusée et signalée."
        ),
        metadata={"source": "politique_securite.md", "categorie": "securite"},
    ),
]
print("Documents :", len(documents))


## Construire deux découpages

Le petit découpage favorise des passages ciblés. Le grand conserve davantage de contexte. Nous gardons le même modèle d'embeddings pour isoler l'effet du chunking.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


petit_splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=35)
grand_splitter = RecursiveCharacterTextSplitter(chunk_size=520, chunk_overlap=80)

petits_chunks = petit_splitter.split_documents(documents)
grands_chunks = grand_splitter.split_documents(documents)

print("Petits chunks :", len(petits_chunks))
print("Grands chunks :", len(grands_chunks))
print("\nExemple petit :", petits_chunks[0].page_content)
print("\nExemple grand :", grands_chunks[0].page_content)


## Indexer dans deux bases éphémères

Les bases restent en mémoire pendant ce notebook. Elles servent uniquement à comparer les configurations.


In [ ]:
import chromadb
from langchain_chroma import Chroma


def creer_store(chunks, nom):
    client = chromadb.EphemeralClient()
    store = Chroma(
        client=client,
        collection_name=nom,
        embedding_function=embeddings,
    )
    store.add_documents(chunks, ids=[f"{nom}-{i}" for i in range(len(chunks))])
    return store


store_petit = creer_store(petits_chunks, "chunks_petits")
store_grand = creer_store(grands_chunks, "chunks_grands")


## Définir les attentes avant de chercher

Nous réutilisons `hit@k` : la source attendue doit apparaître parmi les candidats. Une comparaison honnête utilise exactement les mêmes cas pour les deux bases.


In [ ]:
cas_comparaison = [
    {"question": "Quel portail saisir pour le VPN ?", "source_attendue": "procedure_vpn.md"},
    {"question": "Puis-je donner mon code MFA ?", "source_attendue": "politique_securite.md"},
    {"question": "Que vérifier si le moniteur reste noir ?", "source_attendue": "procedure_ecran.md"},
    {"question": "Comment retirer une feuille bloquée ?", "source_attendue": "procedure_imprimante.md"},
]


### À vous de jouer — ajouter une reformulation

Ajoutez un cas dont la formulation ne reprend pas exactement les mots du document.


In [ ]:
# À vous de jouer — ajoutez une reformulation et sa source attendue.
nouveau_cas = None
if nouveau_cas is not None:
    cas_comparaison.append(nouveau_cas)
print("Cas de comparaison :", len(cas_comparaison))


In [ ]:
def hit_at_k(store, cas, k=2):
    hits = []
    for item in cas:
        docs = store.similarity_search(item["question"], k=k)
        sources = [doc.metadata.get("source") for doc in docs]
        hits.append(item["source_attendue"] in sources)
    return sum(hits) / len(hits)


print(f"Petits chunks · hit@2 : {hit_at_k(store_petit, cas_comparaison):.0%}")
print(f"Grands chunks · hit@2 : {hit_at_k(store_grand, cas_comparaison):.0%}")


## Filtrer avec une métadonnée

Un filtre ne comprend pas mieux la question. Il retire simplement les documents dont l'étiquette ne correspond pas.

Il est utile lorsque la catégorie est connue de façon fiable. Un filtre incorrect peut aussi supprimer la bonne réponse.


In [ ]:
question = "Que faire avant d'ouvrir l'appareil ?"
sans_filtre = store_petit.similarity_search(question, k=3)
avec_filtre = store_petit.similarity_search(
    question,
    k=3,
    filter={"categorie": "materiel"},
)

print("Sans filtre :", [doc.metadata for doc in sans_filtre])
print("Avec filtre :", [doc.metadata for doc in avec_filtre])


## MMR : éviter trois passages presque identiques

**MMR** signifie *Maximal Marginal Relevance*. Il cherche un compromis entre proximité avec la question et diversité entre les résultats.

`fetch_k` indique combien de candidats examiner ; `k` indique combien en conserver.


![Similarité, filtre et MMR conduisent au contexte final](../assets/jour_01/09_options_retrieval.png)

*Chaque option répond à un problème différent et doit être évaluée séparément.*


In [ ]:
retriever_mmr = store_petit.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 8, "lambda_mult": 0.5},
)
docs_mmr = retriever_mmr.invoke("Comment diagnostiquer un problème matériel ?")

for doc in docs_mmr:
    print(doc.metadata, "→", doc.page_content[:120])


## Small-to-large, sans magie

Le **parent document retrieval** recherche avec de petits chunks précis, mais renvoie ensuite le passage parent plus large au LLM.

Cette architecture peut améliorer le contexte, mais elle exige un lien fiable entre chaque enfant et son parent. Elle n'est utile qu'après avoir constaté que les petits chunks retrouvent bien le sujet mais manquent de contexte pour répondre.


## À retenir

- Comparer les configurations sur les mêmes questions.
- Ne pas copier une taille ou un seuil comme vérité universelle.
- Un filtre repose sur une métadonnée exacte.
- MMR cherche davantage de diversité.
- Une technique avancée doit répondre à un échec mesuré.
